# Common Task 1.1 -- Dataset Preprocessing for Symbolic Regression

AI Feynman dataset: Feynman_with_units features + FeynmanEquations.csv targets.
Goal: preprocess and tokenize the target equations, document rationale.

## 1. Download Dataset

In [ ]:
import os, subprocess
from pathlib import Path

DATA_DIR = Path("../data/feynman")
DATA_DIR.mkdir(parents=True, exist_ok=True)

EQUATIONS_URL = "https://space.mit.edu/home/tegmark/aifeynman/FeynmanEquations.csv"
FEATURES_URL = "https://www.dropbox.com/scl/fi/kbi1q63opcgsykik0puzg/Feynman_with_units.tar.gz?rlkey=xqm1mb0vkj7iogao825033ltb&dl=1"

csv_path = DATA_DIR / "FeynmanEquations.csv"
tar_path = DATA_DIR / "Feynman_with_units.tar.gz"

if not csv_path.exists():
    print("Downloading FeynmanEquations.csv...")
    subprocess.run(["wget", "-q", EQUATIONS_URL, "-O", str(csv_path)], check=True)
    print(f"  saved: {csv_path} ({csv_path.stat().st_size} bytes)")
else:
    print(f"Already have {csv_path.name}")

if not tar_path.exists():
    print("Downloading Feynman_with_units.tar.gz...")
    subprocess.run(["wget", "-q", FEATURES_URL, "-O", str(tar_path)], check=True)
    print(f"  saved: {tar_path} ({tar_path.stat().st_size / 1e6:.1f} MB)")
else:
    print(f"Already have {tar_path.name}")

# extract
features_dir = DATA_DIR / "Feynman_with_units"
if not features_dir.exists():
    print("Extracting tarball...")
    subprocess.run(["tar", "xzf", str(tar_path), "-C", str(DATA_DIR)], check=True)
    print("Done")
else:
    print(f"Already extracted: {features_dir}")

## 2. Load and Inspect

In [ ]:
import sys
sys.path.insert(0, os.path.abspath(".."))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from src.data.feynman import load_equations, prepare_feynman_dataset, EquationTokenizer

df = load_equations(csv_path)
print(f"Total equations: {len(df)}")
print(f"Columns: {list(df.columns)}")
df.head(10)

In [ ]:
# inspect the target equations
df, tokenizer, formula_col = prepare_feynman_dataset(csv_path)

print(f"Formula column: '{formula_col}'")
print(f"\nSample equations:")
for i, row in df.head(10).iterrows():
    print(f"  {row.get('Filename', i)}: {row[formula_col]}")

## 3. Tokenization

Word-level tokenizer with math-aware regex rules. Each token is one function call, operator, variable, or numeric literal.

**Why word-level, not character-level?**
Functions like `sqrt`, `sin`, `exp` are single semantic units. Character-level tokenization splits them into meaningless characters (`s`, `q`, `r`, `t`) and inflates sequence length by 3-4x.

**Why not BPE?**
The Feynman equation vocabulary is small and fixed (physics variables, standard math functions, operators). BPE would learn merges that don't respect mathematical semantics -- for example, merging `si` from `sin` with `si` from `sigma`. A hand-crafted regex tokenizer preserves the exact mathematical structure.

**Why not prefix/Polish notation?**
Prefix notation (Lample & Charton, 2019) eliminates parentheses and makes tree structure explicit, which helps for generation tasks. But for preprocessing and analysis, infix notation with explicit parentheses is clearer and maps directly to how the equations are written in the dataset. If a downstream model needs prefix notation, the tokenized infix can be converted.

In [ ]:
print(f"Vocabulary size: {tokenizer.vocab_size}")
print(f"\nFull vocabulary:")
for i in range(tokenizer.vocab_size):
    print(f"  {i:3d}: '{tokenizer.id2token[i]}'")

In [ ]:
# round-trip check on a few equations
print("Tokenization examples:\n")
for i, row in df.head(5).iterrows():
    eq = str(row[formula_col])
    tokens = tokenizer.tokenize(eq)
    ids = tokenizer.encode(eq)
    decoded = tokenizer.decode(ids)
    print(f"  Original:  {eq}")
    print(f"  Tokens:    {tokens}")
    print(f"  IDs:       {ids}")
    print(f"  Decoded:   {decoded}")
    print()

## 4. Dataset Statistics

In [ ]:
print(f"Total equations: {len(df)}")
print(f"Vocabulary size: {tokenizer.vocab_size}")
print(f"\nToken count per equation:")
print(f"  mean:   {df['token_count'].mean():.1f}")
print(f"  median: {df['token_count'].median():.0f}")
print(f"  max:    {df['token_count'].max()}")
print(f"  min:    {df['token_count'].min()}")

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(df["token_count"], bins=20, edgecolor="black", alpha=0.7)
axes[0].set_xlabel("Token count")
axes[0].set_ylabel("Number of equations")
axes[0].set_title("Equation length distribution")

# count unique variables per equation
def count_vars(tokens):
    return len([t for t in tokens if t.isalpha() and t not in
                {"sqrt", "exp", "log", "sin", "cos", "tan", "asin", "acos",
                 "atan", "arcsin", "arccos", "arctan", "tanh", "cosh", "sinh",
                 "pi"}])

df["n_vars"] = df["tokens"].apply(count_vars)
axes[1].hist(df["n_vars"], bins=range(0, df["n_vars"].max() + 2),
             edgecolor="black", alpha=0.7)
axes[1].set_xlabel("Number of variables")
axes[1].set_ylabel("Number of equations")
axes[1].set_title("Variables per equation")

plt.tight_layout()
plt.savefig("../data/feynman/stats.png", dpi=150, bbox_inches="tight")
plt.show()

## 5. Feature Data Inspection

Each equation has a corresponding data file with numerical feature columns. Check that we can load them and verify dimensions.

In [ ]:
from src.data.feynman import load_features

# try loading feature data for a few equations
feature_files = sorted(features_dir.glob("*"))
print(f"Feature files found: {len(feature_files)}")

for f in feature_files[:5]:
    data = np.loadtxt(f)
    print(f"  {f.name}: shape={data.shape}")

## 6. Save Processed Data

In [ ]:
tokenizer.save("../data/feynman/vocab.txt")

# save processed dataframe (drop the token list column for parquet compat)
save_df = df.drop(columns=["tokens"])
save_df.to_csv("../data/feynman/equations_processed.csv", index=False)

print(f"Saved vocab ({tokenizer.vocab_size} tokens) to data/feynman/vocab.txt")
print(f"Saved processed equations to data/feynman/equations_processed.csv")